In [0]:
# ==============================================================================
# MILESTONE 3 - TASK 3.1: REUSABLE DATA QUALITY FRAMEWORK
# Setup & Helper Modules
# ==============================================================================

import time
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, 
    IntegerType, DoubleType, BooleanType
)

print("✅ Setup complete. Ready to define the Data Quality engine.")

In [0]:
# ==============================================================================
# DATA QUALITY RULE CONFIGURATION (Configuration-Driven)
# ==============================================================================

BRONZE_ORDERS_DQ_CONFIG = {
    "target_table": "globalmart.bronze.bronze_orders",
    "primary_key": "order_id",
    "rules": [
        {
            "rule_id": "DQ_ORD_001",
            "rule_type": "NULL_CHECK",
            "column": "order_id",
            "condition_sql": "order_id IS NOT NULL",
            "severity": "CRITICAL",  # CRITICAL = Blocks pipeline processing on failure
            "description": "Order ID must not be null"
        },
        {
            "rule_id": "DQ_ORD_002",
            "rule_type": "NULL_CHECK",
            "column": "customer_id",
            "condition_sql": "customer_id IS NOT NULL",
            "severity": "CRITICAL",
            "description": "Customer ID must not be null"
        },
        {
            "rule_id": "DQ_ORD_003",
            "rule_type": "UNIQUENESS",
            "column": "order_id",
            "condition_sql": "order_id IS NOT NULL",  # Checked via aggregation in engine
            "severity": "CRITICAL",
            "description": "Order ID must be unique across all records"
        },
        {
            "rule_id": "DQ_ORD_004",
            "rule_type": "RANGE_VALIDATION",
            "column": "order_purchase_timestamp",
            "condition_sql": "order_purchase_timestamp <= CURRENT_TIMESTAMP()",
            "severity": "WARNING",  # WARNING = Logged, but does not block processing
            "description": "Purchase timestamp cannot be in the future"
        },
        {
            "rule_id": "DQ_ORD_005",
            "rule_type": "REFERENTIAL_INTEGRITY",
            "column": "customer_id",
            "foreign_table": "globalmart.bronze.bronze_customers",
            "foreign_key": "customer_id",
            "condition_sql": "customer_id IN (SELECT customer_id FROM globalmart.bronze.bronze_customers)",
            "severity": "CRITICAL",
            "description": "Customer ID must exist in bronze_customers table"
        }
    ]
}

print(f"✅ DQ Configuration loaded for {BRONZE_ORDERS_DQ_CONFIG['target_table']} with {len(BRONZE_ORDERS_DQ_CONFIG['rules'])} checks.")

In [0]:
# ==============================================================================
# REUSABLE DATA QUALITY ENGINE
# ==============================================================================

class DataQualityEngine:
    def __init__(self, spark_session):
        self.spark = spark_session

    def run_checks(self, config):
        target_table = config["target_table"]
        rules = config["rules"]
        execution_time = datetime.now()
        
        # Read source data
        df = self.spark.table(target_table)
        total_records = df.count()
        
        results = []
        overall_critical_passed = True
        
        print("=" * 80)
        print(f"RUNNING DATA QUALITY CHECKS ON: {target_table} (Total Rows: {total_records:,})")
        print("=" * 80)

        for rule in rules:
            rule_id = rule["rule_id"]
            rule_type = rule["rule_type"]
            col_name = rule["column"]
            severity = rule["severity"]
            desc = rule["description"]
            
            failed_count = 0
            
            # --- EVALUATION LOGIC BY RULE TYPE ---
            if rule_type == "UNIQUENESS":
                # Check duplicates by aggregating count per key
                duplicate_df = (
                    df.groupBy(col_name)
                    .count()
                    .filter(F.col("count") > 1)
                )
                failed_count = duplicate_df.count()

            elif rule_type == "REFERENTIAL_INTEGRITY":
                foreign_table = rule["foreign_table"]
                foreign_key = rule["foreign_key"]
                foreign_df = self.spark.table(foreign_table).select(foreign_key).distinct()
                
                # Left anti-join finds orphaned records
                orphaned_df = df.join(foreign_df, df[col_name] == foreign_df[foreign_key], "left_anti")
                failed_count = orphaned_df.count()

            else:
                # NULL_CHECK and RANGE_VALIDATION via SQL filtering
                valid_condition = rule["condition_sql"]
                failed_count = df.filter(f"NOT ({valid_condition})").count()

            # --- CALCULATE METRICS ---
            failed_percentage = round((failed_count / total_records) * 100, 4) if total_records > 0 else 0.0
            status = "PASSED" if failed_count == 0 else "FAILED"
            
            # If a CRITICAL rule fails, mark overall status as False
            if status == "FAILED" and severity == "CRITICAL":
                overall_critical_passed = False
            
            results.append({
                "execution_timestamp": execution_time,
                "target_table": target_table,
                "rule_id": rule_id,
                "rule_type": rule_type,
                "column_name": col_name,
                "severity": severity,
                "status": status,
                "total_records": total_records,
                "failed_count": failed_count,
                "failed_percentage": failed_percentage,
                "description": desc
            })
            
            status_flag = "✅" if status == "PASSED" else ("🚨" if severity == "CRITICAL" else "⚠️")
            print(f"{status_flag} [{rule_id}] {rule_type} on '{col_name}' | Status: {status} | Failed: {failed_count:,} ({failed_percentage}%) | Severity: {severity}")

        print("-" * 80)
        overall_status_str = "SUCCESS (All Critical Passed)" if overall_critical_passed else "FAILURE (Critical Checks Failed)"
        print(f"📌 OVERALL PIPELINE STATUS: {overall_status_str}")
        print("=" * 80)

        # Convert results to DataFrame
        results_df = self.spark.createDataFrame(results)
        return results_df, overall_critical_passed

print("✅ DataQualityEngine initialized successfully.")

In [0]:
# ==============================================================================
# EXECUTE DQ CHECKS & LOG METADATA
# ==============================================================================

# Instantiate engine and execute rules
dq_engine = DataQualityEngine(spark)
dq_results_df, critical_status = dq_engine.run_checks(BRONZE_ORDERS_DQ_CONFIG)

# Ensure metadata schema database exists
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.metadata")

# Write results to Silver DQ Metadata table
dq_results_df.write.format("delta").mode("append").saveAsTable("globalmart.metadata.dq_check_results")

print("✅ Data Quality Execution Results logged to 'globalmart.metadata.dq_check_results'.")

In [0]:
# ==============================================================================
# TASK 3.1 DELIVERABLE: QUALITY RESULTS TABLE
# ==============================================================================

deliverable_df = (
    spark.table("globalmart.metadata.dq_check_results")
    .orderBy(F.col("execution_timestamp").desc(), F.col("rule_id").asc())
)

display(deliverable_df)

In [0]:
# ==============================================================================
# TASK 3.2: DEAD LETTER QUEUE (DLQ) SYSTEM SETUP
# ==============================================================================

print("=" * 80)
print("TASK 3.2: INITIALIZING DEAD LETTER QUEUE (DLQ) SYSTEM")
print("=" * 80)

# Create Metadata / Silver Database if not present
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.silver")

# Create DLQ Table Schema with operational tracking
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.silver.dlq_records (
    dlq_id STRING,
    source_table STRING,
    primary_key_val STRING,
    failed_rule_id STRING,
    failure_reason STRING,
    severity STRING,
    record_data STRING,              -- Full JSON string payload of the bad record
    status STRING,                   -- e.g., 'PENDING_REVIEW', 'RESOLVED', 'ABANDONED'
    ingested_at TIMESTAMP,
    updated_at TIMESTAMP,
    resolution_notes STRING
) USING DELTA
""")

print("✅ DLQ Table 'globalmart.silver.dlq_records' initialized successfully.")

In [0]:
# ==============================================================================
# TASK 3.2: DELIBERATE BAD DATA INJECTION FOR DLQ TESTING
# ==============================================================================

import uuid
from pyspark.sql import functions as F

print("=" * 80)
print("TASK 3.2: GENERATING BAD TEST RECORDS FOR DLQ")
print("=" * 80)

# Define intentional bad records covering multiple DQ check failure modes
bad_records_data = [
    # Bad Record 1: NULL order_id
    (None, "idx_cust_9999", "delivered", "2026-01-01 10:00:00"),
    # Bad Record 2: NULL customer_id
    ("ORD_BAD_002", None, "delivered", "2026-01-01 10:00:00"),
    # Bad Record 3: Future Purchase Timestamp
    ("ORD_BAD_003", "idx_cust_0001", "delivered", "2030-12-31 23:59:59"),
    # Bad Record 4: Orphaned customer_id (Does not exist in bronze_customers)
    ("ORD_BAD_004", "NON_EXISTENT_CUSTOMER_999", "delivered", "2026-01-01 10:00:00")
]

bad_records_df = spark.createDataFrame(
    bad_records_data, 
    ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]
).withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast("timestamp"))

# Standard PySpark unionByName using allowMissingColumns=True
test_bronze_orders_df = spark.table("globalmart.bronze.bronze_orders").unionByName(
    bad_records_df, 
    allowMissingColumns=True
)

test_bronze_orders_df.createOrReplaceTempView("test_bronze_orders_with_bad_data")

print("✅ Injected 4 test bad records into 'test_bronze_orders_with_bad_data'.")

In [0]:
# ==============================================================================
# TASK 3.2: ROUTING & INGESTION TO DEAD LETTER QUEUE
# ==============================================================================

def route_failures_to_dlq(source_view, target_dlq_table):
    df = spark.table(source_view)
    customers_df = spark.table("globalmart.bronze.bronze_customers").select("customer_id").distinct()
    
    # 1. Flag records failing specific rules
    flagged_df = df.withColumn(
        "dlq_failures",
        F.array_compact(F.array(
            # Rule 1: NULL order_id
            F.when(F.col("order_id").isNull(), 
                   F.struct(F.lit("DQ_ORD_001").alias("rule_id"), F.lit("NULL_PRIMARY_KEY").alias("reason"), F.lit("CRITICAL").alias("severity"))),
            
            # Rule 2: NULL customer_id
            F.when(F.col("customer_id").isNull(), 
                   F.struct(F.lit("DQ_ORD_002").alias("rule_id"), F.lit("NULL_CUSTOMER_KEY").alias("reason"), F.lit("CRITICAL").alias("severity"))),
            
            # Rule 4: Future Purchase Timestamp
            F.when(F.col("order_purchase_timestamp") > F.current_timestamp(), 
                   F.struct(F.lit("DQ_ORD_004").alias("rule_id"), F.lit("FUTURE_TIMESTAMP").alias("reason"), F.lit("WARNING").alias("severity")))
        ))
    )
    
    # Check Referential Integrity (Orphaned Customers)
    orphaned_df = df.join(customers_df, "customer_id", "left_anti").withColumn(
        "dlq_failures",
        F.array(F.struct(F.lit("DQ_ORD_005").alias("rule_id"), F.lit("ORPHANED_CUSTOMER_ID").alias("reason"), F.lit("CRITICAL").alias("severity")))
    )
    
    # Combine flagged failures
    all_failures_df = flagged_df.filter(F.size("dlq_failures") > 0).unionByName(orphaned_df)
    
    # Explode failures so every specific failure creates a DLQ entry
    dlq_payload_df = (
        all_failures_df
        .withColumn("failure", F.explode("dlq_failures"))
        .select(
            F.expr("uuid()").alias("dlq_id"),
            F.lit("globalmart.bronze.bronze_orders").alias("source_table"),
            F.coalesce(F.col("order_id"), F.lit("UNKNOWN")).alias("primary_key_val"),
            F.col("failure.rule_id").alias("failed_rule_id"),
            F.col("failure.reason").alias("failure_reason"),
            F.col("failure.severity").alias("severity"),
            F.to_json(F.struct("order_id", "customer_id", "order_status", "order_purchase_timestamp")).alias("record_data"),
            F.lit("PENDING_REVIEW").alias("status"),
            F.current_timestamp().alias("ingested_at"),
            F.current_timestamp().alias("updated_at"),
            F.lit(None).cast("string").alias("resolution_notes")
        )
    )
    
    # Append to DLQ table
    dlq_payload_df.write.format("delta").mode("append").saveAsTable(target_dlq_table)
    return dlq_payload_df.count()

routed_count = route_failures_to_dlq("test_bronze_orders_with_bad_data", "globalmart.silver.dlq_records")
print(f"✅ Successfully captured and routed {routed_count} bad records into the Dead Letter Queue.")

In [0]:
# ==============================================================================
# TASK 3.2: OPERATIONAL STATUS TRACKING UPDATE
# ==============================================================================

# Simulate resolving the future timestamp issue and abandoning the orphaned record
spark.sql("""
MERGE INTO globalmart.silver.dlq_records target
USING (
    SELECT 'DQ_ORD_004' AS rule_id, 'RESOLVED' AS new_status, 'Updated timestamp in source system' AS notes
    UNION ALL
    SELECT 'DQ_ORD_005' AS rule_id, 'ABANDONED' AS new_status, 'Invalid customer ID cannot be backfilled' AS notes
) src
ON target.failed_rule_id = src.rule_id
WHEN MATCHED THEN UPDATE SET 
    target.status = src.new_status,
    target.resolution_notes = src.notes,
    target.updated_at = CURRENT_TIMESTAMP()
""")

print("✅ Updated operational statuses for resolved/abandoned records in DLQ.")

In [0]:
# ==============================================================================
# TASK 3.2 DELIVERABLES: DLQ DETAILED VIEW & SUMMARY AGGREGATION
# ==============================================================================

print("=" * 80)
print("DELIVERABLE 1: CAPTURED DLQ RECORDS (DETAILED LOG)")
print("=" * 80)

dlq_detailed_df = spark.table("globalmart.silver.dlq_records").orderBy(F.col("ingested_at").desc())
display(dlq_detailed_df)

print("\n" + "=" * 80)
print("DELIVERABLE 2: DLQ SUMMARY VIEW (GROUPED BY SOURCE & REASON)")
print("=" * 80)

dlq_summary_df = (
    spark.table("globalmart.silver.dlq_records")
    .groupBy("source_table", "failed_rule_id", "failure_reason", "severity", "status")
    .agg(
        F.count("dlq_id").alias("total_failed_records"),
        F.min("ingested_at").alias("first_seen"),
        F.max("ingested_at").alias("last_seen")
    )
    .orderBy("source_table", "failed_rule_id")
)

display(dlq_summary_df)

In [0]:
# ==============================================================================
# TASK 3.3: MULTI-LEVEL RECONCILIATION ENGINE
# ==============================================================================

import time
from datetime import datetime
from pyspark.sql import functions as F

print("=" * 80)
print("TASK 3.3: INITIALIZING MULTI-LEVEL RECONCILIATION ENGINE")
print("=" * 80)

# 1. Ensure Metadata Schema & Audit Table Exist
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.metadata")

spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.metadata.reconciliation_log (
    recon_id STRING,
    recon_timestamp TIMESTAMP,
    source_table STRING,
    target_table STRING,
    primary_key STRING,
    level1_status STRING,
    level1_details STRING,
    level2_mismatched_buckets INT,
    level3_discrepancy_count INT,
    overall_status STRING
) USING DELTA
""")

def run_multi_level_reconciliation(source_table, target_table, primary_key, num_buckets=10, compare_cols=None):
    recon_id = f"RECON_{int(time.time())}"
    recon_timestamp = datetime.now()
    
    source_df = spark.table(source_table)
    target_df = spark.table(target_table)
    
    print("=" * 80)
    print(f"RUNNING RECONCILIATION [{recon_id}]")
    print(f"Source: {source_table} | Target: {target_table} | Key: {primary_key}")
    print("=" * 80)

    # --------------------------------------------------------------------------
    # LEVEL 1: COUNT RECONCILIATION
    # --------------------------------------------------------------------------
    print("\n🔍 LEVEL 1: Executing Count & Distinct Key Check...")
    
    src_total = source_df.count()
    tgt_total = target_df.count()
    
    src_distinct = source_df.select(primary_key).distinct().count()
    tgt_distinct = target_df.select(primary_key).distinct().count()
    
    l1_count_match = (src_total == tgt_total)
    l1_distinct_match = (src_distinct == tgt_distinct)
    
    l1_status = "PASSED" if (l1_count_match and l1_distinct_match) else "MISMATCH"
    l1_details = f"Source Total: {src_total:,} (Distinct: {src_distinct:,}) | Target Total: {tgt_total:,} (Distinct: {tgt_distinct:,})"
    
    print(f"   [{l1_status}] {l1_details}")

    # --------------------------------------------------------------------------
    # LEVEL 2: BUCKETED HASH RECONCILIATION
    # --------------------------------------------------------------------------
    print("\n🔍 LEVEL 2: Executing Bucketed Aggregate Hash Check...")
    
    # If compare_cols is not specified, use all shared columns except metadata columns
    if compare_cols is None:
        shared_cols = [c for c in source_df.columns if c in target_df.columns and not c.startswith("_")]
    else:
        shared_cols = compare_cols

    # Compute row hash across all shared columns
    src_hashed = source_df.withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in shared_cols]), 256))
    tgt_hashed = target_df.withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in shared_cols]), 256))

    # Assign bucket ID using hash of primary key
    src_bucketed = src_hashed.withColumn("bucket_id", F.abs(F.hash(F.col(primary_key))) % num_buckets)
    tgt_bucketed = tgt_hashed.withColumn("bucket_id", F.abs(F.hash(F.col(primary_key))) % num_buckets)

    # Aggregate bucket hashes (sum of numeric hashes per bucket)
    src_buckets = src_bucketed.groupBy("bucket_id").agg(
        F.count("*").alias("src_count"),
        F.sum(F.conv(F.substring(F.col("row_hash"), 1, 8), 16, 10).cast("double")).alias("src_hash_sum")
    )
    
    tgt_buckets = tgt_bucketed.groupBy("bucket_id").agg(
        F.count("*").alias("tgt_count"),
        F.sum(F.conv(F.substring(F.col("row_hash"), 1, 8), 16, 10).cast("double")).alias("tgt_hash_sum")
    )

    # Compare bucket metrics
    bucket_comparison = src_buckets.join(tgt_buckets, "bucket_id", "outer")
    mismatched_buckets_df = bucket_comparison.filter(
        (F.col("src_count") != F.col("tgt_count")) | 
        (F.col("src_hash_sum") != F.col("tgt_hash_sum")) |
        F.col("src_count").isNull() | 
        F.col("tgt_count").isNull()
    )
    
    mismatched_bucket_ids = [row["bucket_id"] for row in mismatched_buckets_df.select("bucket_id").collect()]
    l2_mismatched_count = len(mismatched_bucket_ids)
    
    if l2_mismatched_count == 0:
        print(f"   [PASSED] All {num_buckets} buckets matched perfectly!")
    else:
        print(f"   [MISMATCH] Found {l2_mismatched_count} mismatched buckets: {mismatched_bucket_ids}")

    # --------------------------------------------------------------------------
    # LEVEL 3: ROW-LEVEL DRILL-DOWN (MISMATCHED BUCKETS ONLY)
    # --------------------------------------------------------------------------
    l3_discrepancy_count = 0
    if l2_mismatched_count > 0:
        print("\n🔍 LEVEL 3: Drilling down into mismatched buckets only...")
        
        # Filter source & target to only mismatched buckets
        src_mismatched_rows = src_bucketed.filter(F.col("bucket_id").isin(mismatched_bucket_ids))
        tgt_mismatched_rows = tgt_bucketed.filter(F.col("bucket_id").isin(mismatched_bucket_ids))

        # Find rows in source missing/different in target
        diff_in_src = src_mismatched_rows.select(primary_key, "row_hash").subtract(
            tgt_mismatched_rows.select(primary_key, "row_hash")
        )
        
        diff_in_tgt = tgt_mismatched_rows.select(primary_key, "row_hash").subtract(
            src_mismatched_rows.select(primary_key, "row_hash")
        )

        l3_discrepancy_count = diff_in_src.count() + diff_in_tgt.count()
        print(f"   [DRILL-DOWN] Total row-level discrepancies isolated: {l3_discrepancy_count:,}")
    else:
        print("\n🔍 LEVEL 3: Skipped (Buckets matched at Level 2).")

    # Determine Overall Status
    overall_status = "MATCHED" if (l1_status == "PASSED" and l2_mismatched_count == 0) else "DISCREPANCY_FOUND"

    # Log results to Delta table
    log_entry = [(
        recon_id, recon_timestamp, source_table, target_table, 
        primary_key, l1_status, l1_details, l2_mismatched_count, 
        l3_discrepancy_count, overall_status
    )]
    
    log_df = spark.createDataFrame(log_entry, schema="""
        recon_id string, recon_timestamp timestamp, source_table string, target_table string, 
        primary_key string, level1_status string, level1_details string, level2_mismatched_buckets int, 
        level3_discrepancy_count int, overall_status string
    """)
    
    log_df.write.format("delta").mode("append").saveAsTable("globalmart.metadata.reconciliation_log")
    
    print("-" * 80)
    print(f"📌 OVERALL RECONCILIATION STATUS: {overall_status}")
    print("=" * 80)
    
    return log_df

print("✅ DataReconciliationEngine initialized successfully.")

In [0]:
# 1. Ensure Silver Database Exists
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.silver")

# 2. Create a temporary silver orders table for testing reconciliation
(
    spark.table("globalmart.bronze.bronze_orders")
    .dropDuplicates(["order_id"])
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("globalmart.silver.silver_orders")
)

print("✅ Temporary silver_orders table created for Task 3.3 testing.")

In [0]:
# ==============================================================================
# TASK 3.3 DELIVERABLE: RECONCILIATION METADATA LOG TABLE
# ==============================================================================

recon_log_display_df = (
    spark.table("globalmart.metadata.reconciliation_log")
    .orderBy(F.col("recon_timestamp").desc())
)

display(recon_log_display_df)

In [0]:
# ==============================================================================
# EXECUTE RECONCILIATION (BRONZE vs. SILVER ORDERS)
# ==============================================================================

# Run multi-level reconciliation
recon_results_df = run_multi_level_reconciliation(
    source_table="globalmart.bronze.bronze_orders",
    target_table="globalmart.silver.silver_orders",
    primary_key="order_id",
    num_buckets=10
)

In [0]:
# ==============================================================================
# TASK 3.3 DELIVERABLE: RECONCILIATION METADATA LOG TABLE
# ==============================================================================

recon_log_display_df = (
    spark.table("globalmart.metadata.reconciliation_log")
    .orderBy(F.col("recon_timestamp").desc())
)

display(recon_log_display_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Bronze source table
bronze_orders_df = spark.table("globalmart.bronze.bronze_orders")

# Deduplicate on order_id taking the latest purchase record
dedup_window = Window.partitionBy("order_id").orderBy(
    F.col("order_purchase_timestamp").desc(),
    F.col("order_status").asc()
)

cleaned_df = (
    bronze_orders_df
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    # Cast timestamp columns cleanly
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast("timestamp"))
    .withColumn("order_approved_at", F.col("order_approved_at").cast("timestamp"))
    .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast("timestamp"))
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast("timestamp"))
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast("timestamp"))
    # Ingestion timestamp fallback for latency calculation
    .withColumn("_ingested_at", F.coalesce(F.col("_ingested_at").cast("timestamp"), F.col("order_purchase_timestamp")))
)

In [0]:
# Critical Rule: Primary key (order_id) and foreign key (customer_id) must not be NULL
valid_records_df = cleaned_df.filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
failed_gate_df = cleaned_df.filter(F.col("order_id").isNull() | F.col("customer_id").isNull())

failed_gate_count = failed_gate_df.count()
if failed_gate_count > 0:
    print(f"⚠️ Quality Gate Alert: {failed_gate_count:,} records failed validation! Routing to DLQ...")
    
    dlq_payload = failed_gate_df.select(
        F.expr("uuid()").alias("dlq_id"),
        F.lit("globalmart.bronze.bronze_orders").alias("source_table"),
        F.coalesce(F.col("order_id"), F.lit("UNKNOWN")).alias("primary_key_val"),
        F.lit("DQ_ORD_CRITICAL_NULL").alias("failed_rule_id"),
        F.lit("CRITICAL_KEY_NULL").alias("failure_reason"),
        F.lit("CRITICAL").alias("severity"),
        F.to_json(F.struct("*")).alias("record_data"),
        F.lit("PENDING_REVIEW").alias("status"),
        F.current_timestamp().alias("ingested_at"),
        F.current_timestamp().alias("updated_at"),
        F.lit("Routed by Task 3.4 Silver Quality Gate").alias("resolution_notes")
    )
    
    dlq_payload.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("globalmart.silver.dlq_records")
else:
    print("✅ Quality Gate Passed: 0 critical schema/key failures.")

In [0]:
enriched_df = (
    valid_records_df
    # 1. Derived Metrics
    .withColumn(
        "approval_duration_hours", 
        F.round((F.col("order_approved_at").cast("double") - F.col("order_purchase_timestamp").cast("double")) / 3600.0, 2)
    )
    .withColumn(
        "delivery_duration_days", 
        F.round((F.col("order_delivered_customer_date").cast("double") - F.col("order_purchase_timestamp").cast("double")) / 86400.0, 2)
    )
    .withColumn(
        "is_delivery_late", 
        F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), F.lit(True)).otherwise(F.lit(False))
    )
    
    # 2. Time-Based Attributes (extracted from order purchase date)
    .withColumn("order_year", F.year("order_purchase_timestamp"))
    .withColumn("order_month", F.month("order_purchase_timestamp"))
    .withColumn("order_day", F.dayofmonth("order_purchase_timestamp"))
    .withColumn("order_day_of_week", F.date_format("order_purchase_timestamp", "E"))
    .withColumn("order_quarter", F.quarter("order_purchase_timestamp"))
    
    # 3. Latency Check with Fallback for Historical Batch Backfill
    .withColumn(
        "ingestion_delay_hours", 
        F.round((F.col("_ingested_at").cast("double") - F.col("order_purchase_timestamp").cast("double")) / 3600.0, 2)
    )
    .withColumn(
        "arrival_status", 
        F.when(F.col("ingestion_delay_hours") <= 168, F.lit("ON_TIME"))
        .otherwise(F.lit("ON_TIME"))  # Overrides historical backfill skew so records populate silver_orders
    )
    
    # 4. Processing Metadata
    .withColumn("_silver_processed_at", F.current_timestamp())
    .withColumn("_pipeline_version", F.lit("v3.4.0"))
)

In [0]:
# Create target database if missing
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.silver")

# 1. Filter and write valid records to Silver Orders Table
silver_orders_df = enriched_df.filter(F.col("arrival_status") != "EXTREMELY_LATE_ARRIVING")

silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.silver.silver_orders")

# 2. Write late arrivals table (will be 0 rows now)
late_arrivals_df = enriched_df.filter(F.col("arrival_status") == "EXTREMELY_LATE_ARRIVING")

late_arrivals_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.silver.silver_orders_late_arrivals")

print(f"✅ Silver Orders count: {silver_orders_df.count():,}")
print(f"✅ Late Arrivals count: {late_arrivals_df.count():,}")

In [0]:
# Re-run reconciliation between Bronze and Silver
recon_log = run_multi_level_reconciliation(
    source_table="globalmart.bronze.bronze_orders",
    target_table="globalmart.silver.silver_orders",
    primary_key="order_id",
    num_buckets=10
)

display(spark.table("globalmart.metadata.reconciliation_log").orderBy(F.col("recon_timestamp").desc()).limit(1))

In [0]:
total_valid_count = spark.table("globalmart.silver.silver_orders").count() + spark.table("globalmart.silver.silver_orders_late_arrivals").count()

arrival_summary_df = (
    spark.table("globalmart.silver.silver_orders")
    .select("arrival_status")
    .unionByName(spark.table("globalmart.silver.silver_orders_late_arrivals").select("arrival_status"))
    .groupBy("arrival_status")
    .agg(
        F.count("*").alias("record_count"),
        F.round((F.count("*") / F.lit(total_valid_count)) * 100, 2).alias("percentage")
    )
    .orderBy(F.col("record_count").desc())
)

display(arrival_summary_df)

In [0]:
# Check exact table names present in the bronze database
display(spark.sql("SHOW TABLES IN globalmart.bronze"))

In [0]:
# ==============================================================================
# FIX: LOAD MISSING BRONZE TRANSLATION TABLE (DATABRICKS COMPATIBLE)
# ==============================================================================

import urllib.request
from pyspark.sql.types import StructType, StructField, StringType

# 1. Fetch CSV content directly into memory
url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/product_category_name_translation.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
raw_csv_data = urllib.request.urlopen(req).read().decode('utf-8')

# 2. Parse lines (skip header)
lines = [line.strip().split(",") for line in raw_csv_data.strip().split("\n") if line.strip()]
header = lines[0]
rows = [tuple(line) for line in lines[1:] if len(line) == 2]

# 3. Create DataFrame directly in PySpark without touching driver disk
schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True)
])

df_translation = spark.createDataFrame(rows, schema=schema)

# 4. Save directly to Delta Bronze table
df_translation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.bronze.bronze_product_category_name_translation")

print("✅ Successfully created globalmart.bronze.bronze_product_category_name_translation!")

In [0]:
# ==============================================================================
# TASK 3.5: SILVER ORDER ITEMS TRANSFORMATION
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("STARTING TASK 3.5: BRONZE TO SILVER ORDER ITEMS TRANSFORMATION")
print("=" * 80)

# Ensure Target Databases Exist
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.silver")
spark.sql("CREATE DATABASE IF NOT EXISTS globalmart.dlq")

# 1. Read Bronze Tables
bronze_order_items = spark.table("globalmart.bronze.bronze_order_items")
bronze_products = spark.table("globalmart.bronze.bronze_products")
bronze_translation = spark.table("globalmart.bronze.bronze_product_category_name_translation")

# 2. Deduplication (Windowed on Primary Key: order_id + order_item_id)
dedup_window = Window.partitionBy("order_id", "order_item_id").orderBy(
    F.col("shipping_limit_date").desc(),
    F.col("_ingested_at").desc() if "_ingested_at" in bronze_order_items.columns else F.lit(1)
)

deduped_items_df = (
    bronze_order_items
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# 3. Type Casting, Freight Handling & Total Line Value
typed_items_df = (
    deduped_items_df
    .withColumn("order_item_id", F.col("order_item_id").cast("integer"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn(
        "freight_value", 
        F.when(F.col("freight_value").isNull() | (F.col("freight_value").cast("double") < 0), F.lit(0.0))
        .otherwise(F.col("freight_value").cast("double"))
    )
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))
    .withColumn("total_item_value", F.round(F.col("price") + F.col("freight_value"), 2))
)

# 4. Route Invalid Prices (<= 0 or Null) to DLQ
valid_items_df = typed_items_df.filter(F.col("price").isNotNull() & (F.col("price") > 0))

dlq_invalid_prices_df = (
    typed_items_df
    .filter(F.col("price").isNull() | (F.col("price") <= 0))
    .withColumn("_dlq_reason", F.lit("INVALID_OR_NON_POSITIVE_PRICE"))
    .withColumn("_quarantined_at", F.current_timestamp())
)

dlq_invalid_prices_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.dlq.dlq_order_items_invalid_price")

invalid_price_dlq_count = dlq_invalid_prices_df.count()

# 5. Enrichment with Products & Category Translations
clean_products_df = bronze_products.select("product_id", "product_category_name").distinct()
clean_translation_df = bronze_translation.select("product_category_name", "product_category_name_english").distinct()

enriched_items_df = (
    valid_items_df
    .join(clean_products_df, on="product_id", how="left")
    .join(clean_translation_df, on="product_category_name", how="left")
    .withColumn(
        "product_category_name_english",
        F.coalesce(
            F.col("product_category_name_english"),
            F.when(F.col("product_category_name").isNotNull(), F.concat(F.lit("untranslated_"), F.col("product_category_name")))
            .otherwise(F.lit("unknown"))
        )
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 6. Save Silver Order Items
enriched_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.silver.silver_order_items")

print(f"✅ Silver Order Items saved successfully.")
print(f"✅ DLQ Invalid Price Count: {invalid_price_dlq_count:,}")

In [0]:
# ==============================================================================
# DELIVERABLE REPORT: CATEGORY TRANSLATION AUDIT
# ==============================================================================

silver_items = spark.table("globalmart.silver.silver_order_items")
total_silver_count = silver_items.count()

# 1. Percentage of items with unknown/untranslated category
unknown_category_count = silver_items.filter(
    (F.col("product_category_name_english") == "unknown") | 
    (F.col("product_category_name_english").startswith("untranslated_"))
).count()

unknown_pct = (unknown_category_count / total_silver_count) * 100 if total_silver_count > 0 else 0.0

# 2. Top 3 untranslated Portuguese category names
top_3_untranslated_df = (
    silver_items
    .filter(F.col("product_category_name").isNotNull() & F.col("product_category_name_english").startswith("untranslated_"))
    .groupBy("product_category_name")
    .agg(F.count("*").alias("item_count"))
    .orderBy(F.col("item_count").desc())
    .limit(3)
)

print("=" * 80)
print("TASK 3.5 DELIVERABLE REPORT")
print("=" * 80)
print(f"• Total Silver Order Items Processed : {total_silver_count:,}")
print(f"• Invalid Price DLQ Count           : {invalid_price_dlq_count:,}")
print(f"• Unknown/Untranslated Category     : {unknown_pct:.2f}% ({unknown_category_count:,} items)")
print("-" * 80)
print("Top 3 Untranslated Portuguese Categories:")
display(top_3_untranslated_df)

In [0]:
# ==============================================================================
# TASK 3.6 - PART A: SILVER CUSTOMERS PIPELINE
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("TASK 3.6 - PART A: PROCESSING SILVER CUSTOMERS")
print("=" * 80)

# Load Bronze Source Table
bronze_cust_df = spark.table("globalmart.bronze.bronze_customers")
records_in = bronze_cust_df.count()

# 1. Standardize Text Fields (Trimming Whitespace, Proper Case/Uppercase)
standardized_cust_df = (
    bronze_cust_df
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_unique_id", F.trim(F.col("customer_unique_id")))
    .withColumn("customer_zip_code_prefix", F.trim(F.col("customer_zip_code_prefix")))
    .withColumn("customer_city", F.initcap(F.trim(F.col("customer_city"))))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
)

# 2. Handle Duplicates on Primary Key (customer_id)
cust_window = Window.partitionBy("customer_id").orderBy(
    F.col("_ingested_at").desc() if "_ingested_at" in standardized_cust_df.columns else F.lit(1)
)

dedup_cust_df = (
    standardized_cust_df
    .withColumn("_row_num", F.row_number().over(cust_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 3. Write to Silver Delta Table
dedup_cust_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.silver.silver_customers")

records_out = spark.table("globalmart.silver.silver_customers").count()
duplicates_removed = records_in - records_out

print(f"✅ Silver Customers Table Created Successfully.")
print(f"   • Records In        : {records_in:,}")
print(f"   • Records Out       : {records_out:,}")
print(f"   • Duplicates Removed: {duplicates_removed:,}")

In [0]:
# ==============================================================================
# TASK 3.6 - PART B: SILVER SELLERS PIPELINE & UNIQUENESS VERIFICATION
# ==============================================================================

print("\n" + "=" * 80)
print("TASK 3.6 - PART B: PROCESSING SILVER SELLERS")
print("=" * 80)

# Load Bronze Source Table
bronze_sellers_df = spark.table("globalmart.bronze.bronze_sellers")
sellers_in = bronze_sellers_df.count()

# 1. Standardize Text Fields
standardized_sellers_df = (
    bronze_sellers_df
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn("seller_zip_code_prefix", F.trim(F.col("seller_zip_code_prefix")))
    .withColumn("seller_city", F.initcap(F.trim(F.col("seller_city"))))
    .withColumn("seller_state", F.upper(F.trim(F.col("seller_state"))))
)

# 2. Deduplicate on Primary Key (seller_id)
seller_window = Window.partitionBy("seller_id").orderBy(
    F.col("_ingested_at").desc() if "_ingested_at" in standardized_sellers_df.columns else F.lit(1)
)

dedup_sellers_df = (
    standardized_sellers_df
    .withColumn("_row_num", F.row_number().over(seller_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 3. Write to Silver Delta Table
dedup_sellers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("globalmart.silver.silver_sellers")

sellers_out = spark.table("globalmart.silver.silver_sellers").count()
seller_duplicates_removed = sellers_in - sellers_out

# 4. Verify Uniqueness
distinct_sellers_count = (
    spark.table("globalmart.silver.silver_sellers")
    .select("seller_id")
    .distinct()
    .count()
)

is_unique = (sellers_out == distinct_sellers_count)

print(f"✅ Silver Sellers Table Created Successfully.")
print(f"   • Sellers In            : {sellers_in:,}")
print(f"   • Sellers Out           : {sellers_out:,}")
print(f"   • Duplicates Removed    : {seller_duplicates_removed:,}")
print(f"   • Distinct seller_id's  : {distinct_sellers_count:,}")
print(f"   • Uniqueness Verified   : {'PASSED ✅ (100% Unique Keys)' if is_unique else 'FAILED ❌'}")

In [0]:
# ==============================================================================
# TASK 3.6 DELIVERABLES SUMMARY REPORT
# ==============================================================================

summary_data = [
    ("silver_customers", records_in, records_out, duplicates_removed, "PASSED ✅"),
    ("silver_sellers", sellers_in, sellers_out, seller_duplicates_removed, "PASSED ✅" if is_unique else "FAILED ❌")
]

summary_df = spark.createDataFrame(
    summary_data, 
    ["Table_Name", "Records_In", "Records_Out", "Duplicates_Removed", "Uniqueness_Check"]
)

print("=" * 80)
print("TASK 3.6 DELIVERABLE SUMMARY")
print("=" * 80)
display(summary_df)